# FINLORA DATA CLEANING 
*Description :*
Notebook to consolidate all final data cleaning workflow and save to parquet 

In [15]:
import pandas as pd 
import numpy as np
import os

In [16]:
transaction_df = pd.read_csv('../data_assets/FinLora_Customer_Transaction_Dataset.csv')
transaction_df.shape

(11400, 26)

In [17]:
def clean_transaction_data(df):
    """
    Cleans the transaction data by handling missing values, removing duplicates, and standardizing formats.
    
    Parameters:
    df (pd.DataFrame): The input DataFrame containing transaction data.
    
    Returns:
    pd.DataFrame: The cleaned DataFrame.
    """
    # Drop duplicates
    df_raw = (
    df.drop_duplicates(subset=['transaction_id'], keep='first')
    .sort_values(by='transaction_id')
    .reset_index(drop=True)
    )
    #making sure the duplicates have been dropped
    assert df_raw.transaction_id.duplicated().sum() == 0, "Duplicate transaction_id rows still exist"

    # convert amoun_src to numeric 
    df_raw.amount_src= pd.to_numeric(
    df_raw.amount_src.astype("string").str.replace(",","",regex=False),
    errors="coerce"
    );
    print(f"Unable to parse {df_raw['amount_src'].isna().sum()} amount_src")
    df_raw.amount_src.dtype
    assert df_raw.amount_src.dtype == "Float64", "amount_src is not of type float64"

    # convert timestamp to datetime
    converted_timestamps = pd.to_datetime(df_raw.timestamp, errors="coerce", format="mixed", utc=True)
    df_raw.timestamp = converted_timestamps
    df_raw["timestamp_missing"] = df_raw.timestamp.isna()
    print(f"unparseable timestamps: {int(df_raw['timestamp_missing'].sum())}")
    assert df_raw.timestamp.dtype == "datetime64[us, UTC]", "timestamp is not of type datetime64[ns, UTC]"

    # flag the corrupt batch, then mask the bad values to NaN (keep the rows).
    df_raw["corrupt_record"] = (
    (df_raw.fee < 0) | (df_raw.fee > 100)
    | (df_raw.txn_velocity_1h < 0)
    | (df_raw.ip_risk_score > 1.0)
    | (df_raw.device_trust_score < 0)
    | df_raw.amount_src.lt(0)
    )
    df_raw.fee = df_raw.fee.mask((df_raw.fee < 0) | (df_raw.fee > 100))
    df_raw.txn_velocity_1h = df_raw.txn_velocity_1h.mask(df_raw.txn_velocity_1h < 0)
    df_raw.ip_risk_score = df_raw.ip_risk_score.mask(df_raw.ip_risk_score > 1.0)
    df_raw.device_trust_score = df_raw.device_trust_score.mask(df_raw.device_trust_score < 0)
    df_raw.amount_src = df_raw.amount_src.mask(df_raw.amount_src < 0)


    # Impute the missing amount_usd with the EXACT conversion factor.
    exchange_rate = df_raw.exchange_rate_src_to_dest
    complete = df_raw.amount_usd.notna() & df_raw.amount_src.notna() & exchange_rate.notna()
    check = df_raw.loc[complete]
    factor = check.amount_usd / check.amount_src
    median_rate = pd.DataFrame({
    "median": factor.groupby(check.source_currency).median(),
    }).round(4)
    factor_map = df_raw.source_currency.map(median_rate['median'])
    impute = df_raw.amount_usd.isna() & df_raw.amount_src.notna() & factor_map.notna()
    df_raw.loc[impute, "amount_usd"] = df_raw.loc[impute, "amount_src"] * factor_map[impute]
    assert df_raw.amount_usd.dtype == "float64", "amount_usd is not of type float64 after imputation"
    assert df_raw.amount_usd.isna().sum() <= 3, "There are still missing values in amount_usd after imputation, expected at most 3 missing values"

    df_raw.ip_country = df_raw.ip_country.str.lower().str.strip()
    df_raw.home_country = df_raw.home_country.str.lower().str.strip()
    df_raw.ip_country = df_raw.ip_country.replace({"nan":np.nan,"unknown":np.nan})
    df_raw.home_country = df_raw.home_country.replace({"unknown":np.nan})
    assert len(df_raw.ip_country.unique()) == 4, "Unexpected unique values in ip_country"
    assert len(df_raw.home_country.unique()) == 4, "Unexpected unique values in home_country"

    # fix kyc_tier spelling and casing values to 'standard' 'enhanced', 'low' and 'nan'-> NaN
    df_raw.kyc_tier = df_raw.kyc_tier.str.strip().str.lower()
    df_raw.kyc_tier = df_raw.kyc_tier.replace({'standrd': "standard", 'enhancd': "enhanced", 'unknown': np.nan,'nan': np.nan})
    df_raw.kyc_tier = df_raw.kyc_tier.astype("string")
    assert len(df_raw.kyc_tier.unique()) == 4, "Unexpected unique values in kyc_tier"
    assert df_raw.kyc_tier.dtype == "string", "kyc_tier is not of type string"

    # fix channel spelling and casing values to 'mobile' 'web', 'branch' and 'nan'-> NaN
    df_raw.channel = df_raw.channel.str.strip().str.lower()
    df_raw.channel = df_raw.channel.replace({"mobille":"mobile","unknown":np.nan,"weeb":"web"})
    df_raw.channel = df_raw.channel.astype("string")
    assert len(df_raw.channel.unique()) == 4, "Unexpected unique values in channel"
    assert df_raw.channel.dtype == "string", "channel is not of type string"

    # any remaining missing values in the dataset. Missingness isnt random.
    df_raw["timestamp_missing"] = df_raw.timestamp.isna()
    df_raw["record_incomplete"] = df_raw.isna().any(axis=1)
    assert df_raw.record_incomplete.sum() == df_raw.isna().any(axis=1).sum(), "Mismatch in record_incomplete calculation"
    assert df_raw.timestamp_missing.sum() == df_raw.timestamp.isna().sum(), "Mismatch in timestamp_missing calculation"

    return df_raw

In [18]:
df_cleaned = clean_transaction_data(transaction_df)
df_cleaned.shape

Unable to parse 0 amount_src
unparseable timestamps: 60


(11200, 29)

In [19]:
for c in ["amount_src","amount_usd","fee","txn_velocity_1h","ip_risk_score", "device_trust_score","exchange_rate_src_to_dest"]:
    df_cleaned[c] = df_cleaned[c].astype("float64")
for col in ["fee", "txn_velocity_1h", "ip_risk_score", "device_trust_score", "amount_src"]:
        assert pd.api.types.is_numeric_dtype(df_cleaned[col]), f"{col} is not numeric"

assert df_cleaned.fee.dropna().between(0, 100).all()
assert df_cleaned.txn_velocity_1h.dropna().ge(0).all()
assert df_cleaned.ip_risk_score.dropna().between(0, 1).all()
assert df_cleaned.device_trust_score.dropna().between(0, 1).all()
assert df_cleaned.amount_src.dropna().ge(0).all()

print("all assertions passed | rows:", len(df_cleaned))

all assertions passed | rows: 11200


## Data dictionary

| column | description / cleaning note |
|---|---|
| transaction_id | transaction UUID; 200 identical duplicates removed |
| customer_id | customer UUID; repeats allowed |
| timestamp | transaction time (UTC); 60 unparseable -> NaT |
| home_country | customer home country; normalized {ca, uk, us}; 'unknown'->NaN |
| source_currency | currency of amount_src; {CAD, GBP, USD}; clean |
| dest_currency | destination currency; {CAD,CNY,EUR,GBP,INR,MXN,NGN,PHP,USD}; clean |
| channel | normalized {atm, web, mobile}; typos fixed; 'unknown'->NaN |
| amount_src | amount in source currency; negative rows (corrupt batch) masked to NaN |
| amount_usd | USD amount = amount_src x {USD:1.00, CAD:0.74, GBP:1.25}; 3 unrecoverable NaN |
| fee | transaction fee; masked outside [0, 100] (sentinels -1 / 9999.99) |
| exchange_rate_src_to_dest | market cross-rate; DECOY - never used for amount conversion |
| device_id | device fingerprint UUID; high cardinality; clean |
| new_device | bool: first time on this device; clean; strong fraud signal |
| ip_address | IP address; high cardinality; 305 NaN; left as-is |
| ip_country | country from IP; normalized {ca, uk, us}; 'unknown'/'nan'->NaN |
| location_mismatch | bool: IP country != home country; clean; strong fraud signal |
| ip_risk_score | 0-1 IP risk; >1 masked to NaN |
| kyc_tier | normalized {standard, enhanced, low}; 'unknown'/'nan'->NaN |
| account_age_days | days since account opened; clean; fraud accounts much younger |
| device_trust_score | 0-1 device trust; <0 masked to NaN |
| chargeback_history_count | prior chargebacks; clean; near-label leakage watch |
| risk_score_internal | internal risk score 0-0.9; clean; leakage watch |
| txn_velocity_1h | transactions last 1h; negatives masked to NaN |
| txn_velocity_24h | transactions last 24h; clean |
| corridor_risk | corridor risk 0-0.25; clean |
| is_fraud | TARGET (0/1); never modified |
| corrupt_record | derived bool: sentinel/negative batch |
| timestamp_missing | derived bool: unparseable timestamp (60) |
| record_incomplete | derived bool: any missing value after cleaning (811) |

In [20]:
def save_data(to_path:str, cleaned_frame: pd.DataFrame , filename:str)-> None:
    """
    Save the cleaned  dataframe to a parquet file at the specified path.
    """
    os.makedirs(os.path.dirname(to_path), exist_ok=True)
    full_path = os.path.join(to_path, filename)
    cleaned_frame.to_parquet(full_path, index=False)
    print(f"artifact written: {len(cleaned_frame)} rows")
    print("fraud rate:", cleaned_frame['is_fraud'].mean().round(4))

In [21]:
save_data(filename= "finlora_cleaned.parquet",to_path="../data_assets/cleaned/",cleaned_frame=df_cleaned)

artifact written: 11200 rows
fraud rate: 0.0888
